# Grafici su tasso di cambio

Questa categoria genera o registra grafici su tasso bilaterale, cambio effettivo reale e cambio effettivo nominale. Le fonti alternative aiutano a distinguere definizioni BIS, ECB, OECD e WDI.

Ogni grafico riporta la fonte primaria usata nella generazione e la dicitura `elaborazione Nazareno Lecis`.


## Come vengono generati

Il notebook separa tre cose: la specifica del grafico, la fonte dati e la trasformazione. Ogni specifica dichiara `titolo`, `tipo_grafico`, `anno_inizio`, `ultimo_dato`, fonte primaria, fonti alternative e cosa viene mostrato. `definisci_grafico_da_indicatore_world_bank` usa direttamente un indicatore WDI; `definisci_grafico_da_rapporto_world_bank` combina due serie WDI; `registra_grafico_da_collegare_a_api` mantiene in inventario un grafico per cui la fonte pubblica esiste ma non e ancora mappata nel codice.

Quando il grafico viene generato, `genera_grafici_e_inventario` scarica i dati via API, applica la trasformazione indicata, costruisce il confronto con Italia, min-max, percentili 25-75 e mediana dei paesi avanzati quando disponibile, poi mostra l'inventario e lascia il plot nel notebook.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "analisi").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analisi.utils.grafici import (
    PAESI_MONDO,
    definisci_grafico_da_rapporto_world_bank,
    definisci_grafico_da_indicatore_world_bank,
    genera_grafici_e_inventario,
    registra_grafico_da_collegare_a_api,
)

from sources.api_catalog import FONTI_API


## Fonti disponibili per questa categoria

La tabella sotto elenca le API ufficiali considerate per la categoria. Una stessa variabile puo comparire in piu fonti: il notebook indica una fonte primaria per la generazione, ma mantiene le alternative quando esistono definizioni ufficiali equivalenti o vicine.


In [ ]:
fonti_categoria = ('ECB Data Portal API', 'BIS SDMX API', 'OECD SDMX API', 'World Bank API')
catalogo_fonti_api = pd.DataFrame(
    [
        {
            "fonte": fonte.nome,
            "ente": fonte.ente,
            "aree": ", ".join(fonte.aree),
            "note": fonte.note,
            "endpoint_controllo": fonte.url_controllo,
        }
        for fonte in FONTI_API
        if fonte.nome in fonti_categoria
    ]
)
catalogo_fonti_api


## Specifiche dei grafici

Le righe sotto sono volutamente esplicite: per ogni grafico si legge il titolo dell'analisi, il tipo di grafico, la fonte primaria, le fonti ufficiali alternative, l'anno di partenza, l'ultimo dato richiesto e la trasformazione applicata.


In [ ]:
SPECIFICHE_GRAFICI = [
    registra_grafico_da_collegare_a_api(
        titolo='Tasso bilaterale nominale',
        nome_output='tasso_bilaterale_nominale',
        fonte_primaria='ECB Data Portal API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Tasso bilaterale nominale: Crescita media e cumulata del cambio bilaterale.',
        fonti_alternative=('BIS SDMX API', 'OECD SDMX API'),
        note='Crescita media e cumulata del cambio bilaterale.',
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Tasso multilaterale reale',
        nome_output='tasso_multilaterale_reale',
        indicatore='PX.REX.REER',
        trasformazione='growth_10y',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Tasso multilaterale reale: tasso di crescita annualizzato su 10 anni, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('BIS SDMX API', 'ECB Data Portal API', 'OECD SDMX API'),
    ),
    definisci_grafico_da_indicatore_world_bank(
        titolo='Tasso multilaterale reale - crescita cumulata',
        nome_output='tasso_multilaterale_reale_crescita_cumulata',
        indicatore='PX.REX.REER',
        trasformazione='index',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        fonte_primaria='World Bank API',
        cosa_mostra='Tasso multilaterale reale - crescita cumulata: indice con base 1970=100, con confronto Italia-distribuzione paesi avanzati.',
        fonti_alternative=('BIS SDMX API', 'ECB Data Portal API', 'OECD SDMX API'),
    ),
    registra_grafico_da_collegare_a_api(
        titolo='Tasso multilaterale nominale',
        nome_output='tasso_multilaterale_nominale',
        fonte_primaria='BIS SDMX API',
        tipo_grafico='serie_storica',
        anno_inizio=1960,
        ultimo_dato='latest',
        cosa_mostra='Tasso multilaterale nominale: Crescita media e cumulata del cambio effettivo nominale.',
        fonti_alternative=('OECD SDMX API', 'ECB Data Portal API'),
        note='Crescita media e cumulata del cambio effettivo nominale.',
    ),
]

print(f"Specifiche nella categoria: {len(SPECIFICHE_GRAFICI)}")


## Inventario e generazione

La tabella risultante mostra cosa viene prodotto, quale fonte e stata usata, quali alternative sono possibili, da quale anno parte la serie, fino a quale ultimo dato viene richiesto e quali grafici restano da collegare.


In [ ]:
cartella_output = ROOT / "analisi" / "tasso_cambio"
percorsi, inventario = genera_grafici_e_inventario(SPECIFICHE_GRAFICI, cartella_output)

colonne = [
    "titolo",
    "tipo_grafico",
    "cosa_mostra",
    "fonte_primaria",
    "fonti_alternative",
    "anno_inizio",
    "ultimo_dato",
    "trasformazione",
    "confronto",
    "stato",
    "nome_output",
    "note",
    "errore",
]
print(f"PNG generati: {len(percorsi)}")
for percorso in percorsi:
    print(percorso)
inventario[colonne]


## Plot dei grafici generati

I plot visualizzati qui sono quelli creati dalla cella precedente. I PNG locali restano fuori da Git.


In [ ]:
from IPython.display import Image, display

if not percorsi:
    print("Nessun PNG generato: tutti i grafici della categoria sono ancora da collegare o la fonte non ha restituito dati.")
for percorso in percorsi:
    display(Image(filename=str(percorso)))
